<a href="https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Đối với bài toán phân lớp nhị phân dự đoán sự suy giảm hiệu suất (is_declining_label), chúng ta thiết lập ranh giới quyết định (decision boundary) theo hai giai đoạn. Đầu tiên, ta sử dụng Hồi quy Logistic làm cơ sở toán học tuyến tính để xác định các động lực chính (primary drivers) và kiểm tra tính hội tụ của không gian đặc trưng. Sau đó, ta sử dụng Random Forest để nắm bắt các tương tác đặc trưng phi tuyến và tính dị phương sai mà không cần tạo đặc trưng thủ công phức tạp (manual feature engineering). Sự đơn giản của hồi quy giúp ta có một baseline có khả năng diễn giải cao trước khi chấp nhận sự phức tạp của mô hình cây.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_score, recall_score, accuracy_score
from typing import Tuple, List

# Cố định seed để đảm bảo tính tái tạo toán học của kết quả
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Đọc tập dữ liệu starter
df = pd.read_csv("https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main/data/raw/content_refresh_anonymized.csv")

# Kiểm tra sanity check ban đầu
print(f"Tổng số mẫu: {len(df)} | Số lượng client: {df['client_id'].nunique()}")

Tổng số mẫu: 30000 | Số lượng client: 32


## 2. Split design

Để bảo toàn tính độc lập thống kê, ta áp dụng phương pháp Grouped Split phân tách theo client_id. Các ID ẩn danh như client_id và content_id tuyệt đối không được sử dụng làm đặc trưng (features) huấn luyện. Việc sử dụng GroupShuffleSplit ngăn chặn tình trạng mô hình "học vẹt" (memorize) phân phối dữ liệu riêng biệt của một khách hàng cụ thể, đảm bảo mô hình có khả năng tổng quát hóa trên những khách hàng chưa từng thấy.  Bên cạnh đó, ta phải loại bỏ cạm bẫy nhãn (label trap): is_declining_label được nội suy trực tiếp từ trend_direction và trend_pct. Việc giữ lại hai cột này sẽ gây ra rò rỉ mục tiêu (target leakage) nghiêm trọng ở mức độ toán học[cite: 1].  

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from typing import Tuple

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Xử lý ma trận đặc trưng: thiết lập biến mục tiêu, xử lý cạm bẫy nhãn và các giá trị khuyết thiếu.
    """
    df_clean = data.copy()

    # [SỬA LỖI] 0. Xây dựng biến mục tiêu toán học TRƯỚC KHI loại bỏ cột rò rỉ
    # Theo định nghĩa, sự suy giảm (declining) xảy ra khi phần trăm xu hướng mang giá trị âm[cite: 1].
    if 'trend_pct' in df_clean.columns:
        df_clean['is_declining_label'] = (df_clean['trend_pct'] < 0).astype(int)
    elif 'trend_direction' in df_clean.columns:
        df_clean['is_declining_label'] = df_clean['trend_direction'].astype(str).str.lower().str.contains('declin|down').astype(int)
    else:
        raise ValueError("Không tìm thấy dữ liệu về trend để khởi tạo nhãn is_declining_label.")

    # 1. Loại bỏ các đặc trưng rò rỉ (Label Trap) và các ID
    leakage_cols = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
    id_cols = ['content_id']
    drop_cols = leakage_cols + id_cols
    df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

    # 2. Xử lý khuyết thiếu sinh ra tín hiệu danh mục (category signal)[cite: 1]
    # Thay vì fillna(0) một cách mù quáng, ta thêm cờ (has_-flags) để bảo toàn không gian thông tin[cite: 1]
    if 'word_count' in df_clean.columns:
        df_clean['has_word_count'] = df_clean['word_count'].notna().astype(int)
        df_clean['word_count'] = df_clean['word_count'].fillna(df_clean['word_count'].median())

    # Mã hóa One-Hot cho các biến phân loại, ngoại trừ client_id
    cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
    if 'client_id' in cat_cols:
        cat_cols.remove('client_id')

    df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
    return df_clean

def create_honest_split(df: pd.DataFrame, group_col: str = 'client_id', test_size: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Phân tách tập dữ liệu đảm bảo sự độc lập theo nhóm client_id."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42) # RANDOM_SEED = 42
    train_idx, test_idx = next(gss.split(df, groups=df[group_col]))
    return df.iloc[train_idx], df.iloc[test_idx]

# Chuẩn bị dữ liệu (giả định bạn đã có biến df từ Cell 2)
df_features = engineer_features(df)
train_df, test_df = create_honest_split(df_features)

X_train = train_df.drop(columns=['is_declining_label', 'client_id'])
y_train = train_df['is_declining_label']
X_test = test_df.drop(columns=['is_declining_label', 'client_id'])
y_test = test_df['is_declining_label']

print(f"Kích thước ma trận đặc trưng Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Phân phối nhãn Train:\n{y_train.value_counts(normalize=True)}")

Kích thước ma trận đặc trưng Train: (23837, 59) | Test: (6163, 59)
Phân phối nhãn Train:
is_declining_label
1    0.664765
0    0.335235
Name: proportion, dtype: float64


/tmp/ipykernel_7197/1990300317.py:34: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()


## 3. Train + compare vs my baseline

Bảng so sánh dưới đây đối chiếu ranh giới quyết định của mô hình máy học với một Heuristic Baseline từ Tuần 4. Quy tắc thiết yếu là: cùng một tập dữ liệu, cùng một tập phân tách (split), và cùng các độ đo (metrics). Nếu một mô hình phức tạp hơn không mang lại sự vượt trội đáng kể về Precision/Recall so với Logistic Regression, ta sẽ ưu tiên sự đơn giản để giữ nguyên khả năng diễn giải.

### Phân tích hiệu suất trung thực (Honest Performance)

Sau khi loại bỏ `trend_direction`, `trend_pct`, `impressions_last_30d` và `impressions_prev_30d`
(các cột dẫn xuất trực tiếp ra nhãn), điểm số rơi từ mức "hoàn hảo đáng ngờ" xuống mức thật —
và mức thật thì khiêm tốn. Đọc bảng bên dưới **cùng với base rate**, vì thiếu base rate thì mọi
con số accuracy đều vô nghĩa:

* **Base rate của tập test = 0.6278.** Đoán bừa "mọi trang đều suy giảm" đã cho Accuracy 0.628
  và Recall 1.000. Đó mới là mốc phải vượt qua, không phải con số 0.
* **Logistic Regression:** Accuracy 0.668 / Precision 0.694 / Recall 0.842. So với việc đoán bừa
  theo lớp đa số, mô hình chỉ hơn **4.0 điểm** accuracy và **6.6 điểm** precision. Có kỹ năng, nhưng
  ít — và nó *mất* recall (0.842 so với 1.000) để đổi lấy precision đó.
* **Random Forest:** Accuracy 0.664 / Precision 0.682 / Recall 0.871. Phức tạp hơn nhưng không
  vượt được Logistic Regression ở cả accuracy lẫn precision.
* **Rule Baseline (W4) ở đây là một baseline yếu:** quy tắc `ctr < trung vị` đạt Accuracy 0.461 —
  *thấp hơn cả việc đoán bừa*. Vượt qua nó không chứng minh được gì nhiều; đó là lý do capstone
  sau này dựng lại baseline nghiêm túc hơn và chuyển hẳn sang precision@K.
* **Quyết định:** chọn Logistic Regression — không phải vì nó mạnh, mà vì ở mức chênh lệch nhỏ
  như thế này thì khả năng đọc được trọng số có giá trị hơn vài phần nghìn điểm số.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
from typing import List
import numpy as np
import pandas as pd

# Baseline Tuần 4 (bản mô phỏng bằng CTR)
baseline_threshold = X_train['ctr'].median()
y_pred_baseline = (X_test['ctr'] < baseline_threshold).astype(int)

# Sàn dưới của mọi sàn: đoán theo lớp đa số. Không có dòng này thì accuracy không đọc được.
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

# Chuẩn hóa không gian đặc trưng (Rất quan trọng cho Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.fillna(0))
X_test_scaled = scaler.transform(X_test.fillna(0))

# Khởi tạo và tối ưu các mô hình trên dữ liệu đã chuẩn hóa
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

# Hàm đánh giá — thêm cột tỷ lệ dự đoán dương để thấy mô hình có đang "đoán bừa lớp đa số" không
def evaluate_model(y_true: np.ndarray, preds: List[np.ndarray], model_names: List[str]) -> pd.DataFrame:
    results = []
    for p, name in zip(preds, model_names):
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_true, p),
            'Precision': precision_score(y_true, p, zero_division=0),
            'Recall': recall_score(y_true, p, zero_division=0),
            'Tỷ lệ dự đoán "suy giảm"': float(np.mean(p)),
        })
    return pd.DataFrame(results).set_index('Model')

print(f"Base rate tập test (tỷ lệ trang thực sự có nhãn suy giảm): {y_test.mean():.4f}")
print(f"Cỡ tập test: {len(y_test):,} dòng / {test_df['client_id'].nunique()} client chưa từng thấy\n")

# Bảng so sánh — mọi dòng cùng dữ liệu, cùng split, cùng độ đo
comparison_table = evaluate_model(
    y_test,
    [y_pred_dummy, y_pred_baseline, y_pred_lr, y_pred_rf],
    ['Đoán lớp đa số (sàn)', 'Rule Baseline (W4)', 'Logistic Regression', 'Random Forest']
)
display(comparison_table.round(4))

print(f"Kỹ năng thật của Logistic Regression so với việc đoán bừa: "
      f"{(accuracy_score(y_test, y_pred_lr) - accuracy_score(y_test, y_pred_dummy)) * 100:+.1f} điểm accuracy, "
      f"{(precision_score(y_test, y_pred_lr) - precision_score(y_test, y_pred_dummy)) * 100:+.1f} điểm precision.")

Base rate tập test (tỷ lệ trang thực sự có nhãn suy giảm): 0.6278
Cỡ tập test: 6,163 dòng / 7 client chưa từng thấy



,Accuracy,Precision,Recall,"Tỷ lệ dự đoán ""suy giảm"""
Model,,,,
Đoán lớp đa số (sàn),0.6278,0.6278,1.0000,1.0000
Rule Baseline (W4),0.4611,0.5811,0.5074,0.5481
Logistic Regression,0.6679,0.6942,0.8416,0.7610
Random Forest,0.6644,0.6825,0.8705,0.8007


Kỹ năng thật của Logistic Regression so với việc đoán bừa: +4.0 điểm accuracy, +6.6 điểm precision.


In [4]:
import pandas as pd
import numpy as np

# Giữ DẤU của hệ số, không chỉ giá trị tuyệt đối: dấu mới cho biết tín hiệu đẩy rủi ro
# lên hay xuống, và ở đây nó phơi bày một cặp cột gần như triệt tiêu nhau.
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Hệ số (đã chuẩn hóa)': log_reg.coef_[0],
})
coef_df['|Hệ số|'] = coef_df['Hệ số (đã chuẩn hóa)'].abs()
coef_df = coef_df.sort_values('|Hệ số|', ascending=False)

print("Top 6 đặc trưng có ảnh hưởng mạnh nhất trong Logistic Regression:")
display(coef_df.head(6).round(3))

print("Lưu ý: clicks_last_30d và clicks_90d có hệ số gần bằng nhau nhưng NGƯỢC DẤU "
      "(-2.66 và +2.60). Đây là hai cột cộng tuyến gần như triệt tiêu nhau, "
      "nên không được đọc từng hệ số như một tác động độc lập.")

Top 6 đặc trưng có ảnh hưởng mạnh nhất trong Logistic Regression:


,Feature,Hệ số (đã chuẩn hóa),|Hệ số|
15,clicks_last_30d,-2.661,2.661
6,clicks_90d,2.603,2.603
13,days_with_impressions,1.040,1.040
18,sessions_prev_30d,0.964,0.964
12,scroll_events_90d,0.889,0.889
14,days_with_sessions,-0.542,0.542


Lưu ý: clicks_last_30d và clicks_90d có hệ số gần bằng nhau nhưng NGƯỢC DẤU (-2.66 và +2.60). Đây là hai cột cộng tuyến gần như triệt tiêu nhau, nên không được đọc từng hệ số như một tác động độc lập.


**Đọc bảng trên cho đúng — và điều nó *không* nói:**

* **Không có con số nào "hoàn hảo đáng ngờ" ở đây.** Việc loại `trend_pct`, `trend_direction`,
  `impressions_last_30d` và `impressions_prev_30d` đã cắt đứt đường rò rỉ trực tiếp — nếu chúng còn
  nằm trong ma trận $X$ thì điểm số sẽ bật lên gần 1.0 (bài test cố ý nhét lại `trend_pct` nằm ở
  `w03_feature_leakage_check.ipynb`, cho ROC-AUC 0.9997).
* **Nhưng "không hoàn hảo" chưa phải là "sạch".** Feature set này vẫn còn `clicks_90d`,
  `sessions_90d`, `ctr`, `avg_position`, `days_with_impressions` — mọi tổng 90 ngày đều **bao trùm
  cửa sổ 30 ngày cuối**, tức là chính cửa sổ sinh ra nhãn. Nó vẫn còn `position_tier_*` và
  `impression_tier_*`, là các bucket quyết định của sản phẩm. Đây là **rò rỉ chồng lấn cửa sổ**, kín
  đáo hơn nhiều so với loại rò rỉ trực tiếp, và nó là lý do capstone dựng lại toàn bộ hợp đồng đặc
  trưng bằng cách tách cửa sổ 90 ngày thành `first30 + prev30 + last30`.
* **Cả hai mô hình đang nghiêng mạnh về lớp đa số.** Logistic Regression gán nhãn "suy giảm" cho
  76.1% số trang, Random Forest 80.1%, trong khi thực tế là 62.8%. Recall cao (0.84–0.87) chủ yếu
  đến từ việc đoán "có" nhiều hơn, không phải từ khả năng phân biệt sắc bén.
* **Random Forest không mua được gì bằng độ phức tạp** (0.664 so với 0.668 accuracy). Theo nguyên
  tắc "đơn giản là một tính năng", mô hình được chọn là Logistic Regression.

**Kết luận:** điểm số này **tin được ở mức đã đo**, nhưng nó chưa trả lời đúng câu hỏi nghiệp vụ.
Accuracy trên toàn tập không phải là thứ một biên tập viên dùng — họ cần biết trong 50 trang đứng
đầu danh sách có bao nhiêu trang thật sự đang tụt. Đó là lý do capstone chuyển hẳn sang
**precision@K + PR-AUC**, và siết lại feature set theo đúng thời điểm ra quyết định.

## 4. Errors and interpretation

Một độ đo vô hướng (scalar metric) không nói gì về *hình dạng* của lỗi. Ở mục này tôi làm hai việc
thật, thay vì in lại bảng cũ:

1. **Permutation importance** trên tập test — xáo trộn từng cột và đo accuracy rơi bao nhiêu. Nếu có
   một cột nào đó "gánh" toàn bộ mô hình thì đó là chỗ đầu tiên phải nghi ngờ rò rỉ.
2. **Ba trường hợp False Positive mà mô hình tự tin nhất** — mô hình khẳng định "trang này đang suy
   giảm" với xác suất cao nhất, nhưng nhãn thực tế là không suy giảm. Sai ở nơi mình chắc chắn nhất
   là chỗ lộ ra giả định sai.

In [5]:
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

# --- (1) Permutation importance: cột nào thực sự gánh mô hình? ---
perm = permutation_importance(log_reg, X_test_scaled, y_test, n_repeats=5,
                              random_state=RANDOM_SEED, scoring="accuracy", n_jobs=-1)
perm_df = (pd.DataFrame({'Feature': X_train.columns,
                         'Accuracy rơi khi xáo trộn': perm.importances_mean,
                         'Độ lệch chuẩn': perm.importances_std})
           .sort_values('Accuracy rơi khi xáo trộn', ascending=False))
print("Permutation importance (Logistic Regression, đo trên tập test):")
display(perm_df.head(6).round(5))

# --- (2) Ba False Positive mà mô hình tự tin nhất ---
prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
is_fp = (y_pred_lr == 1) & (y_test.values == 0)
is_fn = (y_pred_lr == 0) & (y_test.values == 1)
print(f"\nSố False Positive: {int(is_fp.sum()):,} | False Negative: {int(is_fn.sum()):,} "
      f"trên {len(y_test):,} dòng test")

fp_pos = np.where(is_fp)[0]
top_fp = fp_pos[np.argsort(-prob_lr[fp_pos])[:3]]
raw_test = df.loc[test_df.index]          # dòng gốc, để đọc được cả trend_pct thật
cols = ['impressions_90d', 'clicks_90d', 'clicks_last_30d', 'days_with_impressions',
        'avg_position', 'ctr', 'trend_pct']
fp_view = raw_test.iloc[top_fp][cols].copy()
fp_view.insert(0, 'Xác suất mô hình gán "suy giảm"', prob_lr[top_fp].round(3))
print("\n3 trường hợp mô hình SAI mà lại tự tin nhất (thực tế: KHÔNG suy giảm):")
display(fp_view)

print(f"\nĐể so sánh, trung vị của toàn tập test: impressions_90d = "
      f"{raw_test['impressions_90d'].median():,.0f}, days_with_impressions = "
      f"{raw_test['days_with_impressions'].median():.0f}")

Permutation importance (Logistic Regression, đo trên tập test):


,Feature,Accuracy rơi khi xáo trộn,Độ lệch chuẩn
13,days_with_impressions,0.11222,0.00384
15,clicks_last_30d,0.02820,0.00217
58,position_tier_top_3,0.01655,0.00123
14,days_with_sessions,0.01386,0.00282
6,clicks_90d,0.01240,0.00158
23,avg_position,0.00993,0.00438



Số False Positive: 1,434 | False Negative: 613 trên 6,163 dòng test

3 trường hợp mô hình SAI mà lại tự tin nhất (thực tế: KHÔNG suy giảm):


,"Xác suất mô hình gán ""suy giảm""",impressions_90d,clicks_90d,clicks_last_30d,days_with_impressions,avg_position,ctr,trend_pct
21088,0.999,64718,192,40,88,6.0,0.30,15.8
2346,0.984,123561,506,135,88,2.8,0.41,19.9
27750,0.976,94767,312,74,88,3.4,0.33,1.8



Để so sánh, trung vị của toàn tập test: impressions_90d = 429, days_with_impressions = 74


**Mô hình sai ở đâu — và vì sao nó sai ở đó:**

* **Nó dựa gần như hoàn toàn vào một cột duy nhất.** Xáo trộn `days_with_impressions` làm accuracy
  rơi **0.112**, gấp bốn lần cột đứng thứ hai (`clicks_last_30d`, 0.028). Nhưng
  `days_with_impressions` đếm số ngày có hiển thị **trên toàn bộ 90 ngày**, tức là bao gồm cả cửa sổ
  sinh ra nhãn. Cột "quan trọng nhất" của mô hình chính là một cột chồng lấn cửa sổ — đây là phát
  hiện đáng giá nhất của tuần này, và là lý do trực tiếp dẫn tới thiết kế lại ở capstone.
* **Ba lỗi tự tin nhất đều là những trang LỚN và KHỎE.** Cả ba đều có 88/90 ngày có hiển thị, từ
  64,718 đến 123,561 impressions (trung vị toàn tập test chỉ khoảng vài trăm), vị trí trung bình
  2.8–6.0, và `trend_pct` thực tế **dương** (+15.8%, +19.9%, +1.8%) — tức là chúng đang *tăng*.
  Mô hình vẫn gán nhãn "suy giảm" với xác suất 0.976–0.999.
* **Giả định sai nằm ở đâu:** mô hình đã học rằng *"trang càng lớn và càng hiển thị đều thì càng
  thuộc nhóm suy giảm"*. Điều đó đúng trong danh mục của các client ở tập train — nơi phần lớn trang
  lớn đang tụt — nhưng nó không phải một quy luật về nội dung. Khi gặp client mới có trang lớn đang
  tăng trưởng, mô hình sai đúng ở những trang đắt giá nhất.
* **Hệ quả cho quyết định nghiệp vụ:** với 1,434 False Positive trên 6,163 dòng test, danh sách này
  chưa dùng được để giao việc cho biên tập viên. Nó không sai ở chỗ "thiếu chính xác vài phần trăm",
  mà sai ở chỗ **đưa nhầm những trang quan trọng nhất lên đầu**. Đó là lý do capstone bỏ hẳn accuracy,
  chuyển sang xếp hạng theo precision@K, và chỉ giữ lại các cột biết được tại thời điểm ra quyết định.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.